# Preprocessing ELM Qualification data

Clara Krämer, May 2025

### Notebook purpose:
1.


##### 1. Set-up and load raw files

In [1]:
# ───────────────────────────────────────────────────────────────────────────────
# STEP 1: STREAM EACH XML INTO ITS OWN PARQUET FILE
# ───────────────────────────────────────────────────────────────────────────────
import os, glob
from lxml import etree
import pyarrow as pa
import pyarrow.parquet as pq

BASE_PATH = "/Users/go82gax/Documents/Projekte/LFS/Analyse/QUAL Daten/aggregated-datasets_LOQ_20250520"
os.chdir(BASE_PATH)

xml_files = sorted(glob.glob("content *.xml"))
OUT_DIR = "parquet_chunks"
os.makedirs(OUT_DIR, exist_ok=True)

RECORD_TAG = "{http://www.w3.org/1999/02/22-rdf-syntax-ns#}Description"

for fp in xml_files:
    rows = []
    for _, elem in etree.iterparse(fp, events=("end",), tag=RECORD_TAG):
        rows.append({child.tag: child.text for child in elem})
        elem.clear()
    if rows:
        tbl = pa.Table.from_pylist(rows)
        out_fp = os.path.join(OUT_DIR, os.path.basename(fp).replace(".xml", ".parquet"))
        pq.write_table(tbl, out_fp, compression="snappy")
        print(f"✔ Wrote {len(rows)} rows → {out_fp}")
    else:
        print(f"– Skipped empty {fp}")

✔ Wrote 3351402 rows → parquet_chunks/content 10.parquet
✔ Wrote 96063 rows → parquet_chunks/content 11.parquet
✔ Wrote 735048 rows → parquet_chunks/content 12.parquet
✔ Wrote 1098346 rows → parquet_chunks/content 13.parquet
✔ Wrote 72811 rows → parquet_chunks/content 14.parquet
✔ Wrote 132417 rows → parquet_chunks/content 15.parquet
✔ Wrote 2287 rows → parquet_chunks/content 16.parquet
✔ Wrote 11739 rows → parquet_chunks/content 17.parquet
✔ Wrote 33757 rows → parquet_chunks/content 18.parquet
✔ Wrote 106973 rows → parquet_chunks/content 19.parquet
– Skipped empty content 2.xml
✔ Wrote 924237 rows → parquet_chunks/content 20.parquet
– Skipped empty content 21.xml
✔ Wrote 72109 rows → parquet_chunks/content 22.parquet
– Skipped empty content 23.xml
✔ Wrote 54644 rows → parquet_chunks/content 24.parquet
✔ Wrote 636713 rows → parquet_chunks/content 25.parquet
✔ Wrote 5822 rows → parquet_chunks/content 26.parquet
✔ Wrote 612 rows → parquet_chunks/content 27.parquet
✔ Wrote 415 rows → parq

In [2]:
# ───────────────────────────────────────────────────────────────────────────────
# STEP 2: READ ALL PARQUET CHUNKS AS ONE TABLE & INSPECT
# ───────────────────────────────────────────────────────────────────────────────
import pyarrow.dataset as ds
import pandas as pd

# point to your folder of .parquet files
dataset = ds.dataset("parquet_chunks", format="parquet")

# this reads metadata only; very fast
print(dataset)

# convert to pandas (this is fast because Parquet is columnar & compressed)
df = dataset.to_table().to_pandas()

print("\n── Combined shape ──")
print(df.shape)

print("\n── dtypes & non-null counts ──")
print(df.info())

print("\n── Unique & null counts ──")
stats = pd.concat([df.nunique(dropna=False).rename("unique_count"),
                   df.isna().sum().rename("null_count")], axis=1)
print(stats)

print("\n── Top 5 frequent values per column ──")
for col in df.columns:
    print(f"{col!r}:", df[col].value_counts(dropna=False).head(5), "\n")


── Combined shape ──
(26914461, 2)

── dtypes & non-null counts ──
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26914461 entries, 0 to 26914460
Data columns (total 2 columns):
 #   Column                                             Dtype 
---  ------                                             ----- 
 0   {http://www.w3.org/1999/02/22-rdf-syntax-ns#}type  object
 1   {http://data.europa.eu/snb/model/elm/}noteLiteral  object
dtypes: object(2)
memory usage: 410.7+ MB
None

── Unique & null counts ──
                                                   unique_count  null_count
{http://www.w3.org/1999/02/22-rdf-syntax-ns#}type             1    26914461
{http://data.europa.eu/snb/model/elm/}noteLiteral       1737549    24943362

── Top 5 frequent values per column ──
'{http://www.w3.org/1999/02/22-rdf-syntax-ns#}type': {http://www.w3.org/1999/02/22-rdf-syntax-ns#}type
None    26914461
Name: count, dtype: int64 

'{http://data.europa.eu/snb/model/elm/}noteLiteral': {http://data.europa.eu

##### 2. Inspect raw files

In [3]:
# ───────────────────────────────────────────────────────────────────────────────
# INSPECTION: schema, head, samples, null rates
# ───────────────────────────────────────────────────────────────────────────────
import pandas as pd
import pyarrow.dataset as ds

# 0) Reload via dataset (fast)
dataset = ds.dataset("parquet_chunks", format="parquet")
df = dataset.to_table().to_pandas()

# 1) See full schema
print("Arrow schema:")
print(dataset.schema, "\n")

# 2) Rename columns to strip namespaces for convenience
df = df.rename(columns=lambda col: col.split("}")[-1])

# 3) Quick peek
print("DataFrame shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head(), "\n")

# 4) How many non-null noteLiteral?
print("noteLiteral non-null count:", df['noteLiteral'].notna().sum())
print("noteLiteral null count:   ", df['noteLiteral'].isna().sum(), "\n")

# 5) Sample non-null entries
print("Random sample of noteLiteral (5):")
print(df.loc[df['noteLiteral'].notna(), 'noteLiteral']
         .sample(5, random_state=42)
         .tolist(), "\n")

# 6) Top 10 most common notes
print("Top 10 noteLiteral frequencies:")
print(df['noteLiteral']
        .value_counts(dropna=True)
        .head(10), "\n")

Arrow schema:
{http://www.w3.org/1999/02/22-rdf-syntax-ns#}type: null
{http://data.europa.eu/snb/model/elm/}noteLiteral: string 

DataFrame shape: (26914461, 2)

First 5 rows:
   type                                        noteLiteral
0  None  http://data.europa.eu/europassResource/c615416...
1  None                                               None
2  None  http://data.europa.eu/europassResource/b2f4677...
3  None                                               None
4  None  http://data.europa.eu/europassResource/668aa7e... 

noteLiteral non-null count: 1971099
noteLiteral null count:    24943362 

Random sample of noteLiteral (5):
['http://data.europa.eu/europassResource/00fa517c-a73c-41f7-a023-e1a59819c1cf', 'Wichtige Ausbildungsinhalte:\n\nHöhere Lehranstalten für wirtschaftliche Berufe vermitteln Kenntnisse und Fertigkeiten für Berufe in den Bereichen Wirtschaft, Verwaltung, Tourismus und Ernährung. Neben allgemein bildenden Unterrichtsfächern (Deutsch, Mathematik, Fremdsprachen us

In [4]:
# strip the namespace off for convenience
df = df.rename(columns=lambda c: c.split("}")[-1])

# 1. Distribution of text lengths
df['text_len'] = df['noteLiteral'].fillna("").str.len()
print("Text length summary:")
print(df['text_len'].describe(), "\n")

# 2. Histogram of lengths (binned)
print("Length bins:")
print(pd.cut(df['text_len'],
             bins=[0,50,200,500,1000,5000,10000,50000]).value_counts().sort_index(), "\n")

# 3. Sample long entries (>2000 chars)
longs = df[df['text_len'] > 2000]['noteLiteral']
print("Examples of very long notes (3):")
for txt in longs.sample(3, random_state=1):
    print("-", txt[:200].replace("\n"," "), "…\n")

Text length summary:
count    2.691446e+07
mean     1.044322e+01
std      8.513324e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.457900e+04
Name: text_len, dtype: float64 

Length bins:
text_len
(0, 50]             30436
(50, 200]         1730834
(200, 500]         113842
(500, 1000]         55530
(1000, 5000]        40082
(5000, 10000]         358
(10000, 50000]         17
Name: count, dtype: int64 

Examples of very long notes (3):
- Assessments  range from assignments, to presentations to MCQ Tests.   Assignments: Students shall be required to complete 2,000-word assignments  Presentations: carry out 15-minute presentations as pa …

- Unternehmen und Recht:  Der/die Absolvent/Absolventin ist in der Lage, wichtige rechtliche Grundlagen (Unternehmensrecht, Steuerrecht, Arbeits- und Sozialrecht etc.) für unternehmerische und private E …

- <p>Den här praktikkursen vänder sig till dig som läst kurser inom humaniora och teologi och

In [6]:
# ───────────────────────────────────────────────────────────────────────────────
# INSPECTION: one record's full structure
# ───────────────────────────────────────────────────────────────────────────────

from lxml import etree
import glob, os

# 1) Point to your folder & pick the first file
BASE_PATH = "/Users/go82gax/Documents/Projekte/LFS/Analyse/QUAL Daten/aggregated-datasets_LOQ_20250520"
os.chdir(BASE_PATH)
fp = sorted(glob.glob("content *.xml"))[0]

# 2) Parse and locate the first <rdf:Description>
tree = etree.parse(fp)
ns = {"rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#"}
desc = tree.find(".//rdf:Description", namespaces=ns)

# 3) Print its ID
qual_id = desc.get("{http://www.w3.org/1999/02/22-rdf-syntax-ns#}about")
print("Qualification ID (rdf:about):", qual_id, "\n")

# 4) List all child elements, with attributes and text preview
print("Child elements:")
for child in desc:
    tag = child.tag.split("}")[-1]
    text = child.text or "<None>"
    text_preview = text.replace("\n"," ")[:100] + ("…" if len(text) > 100 else "")
    print(f" • {tag}")
    if child.attrib:
        print(f"     Attributes: {child.attrib}")
    print(f"     Text:      {text_preview}\n")

Qualification ID (rdf:about): http://data.europa.eu/snb/data/europassResource/45826aef-a47d-4b9c-b8ce-77d37c837ce2 

Child elements:
 • type
     Attributes: {'{http://www.w3.org/1999/02/22-rdf-syntax-ns#}resource': 'http://data.europa.eu/snb/model/elm/Note'}
     Text:      <None>

 • noteLiteral
     Attributes: {'{http://www.w3.org/XML/1998/namespace}lang': 'en'}
     Text:      http://data.europa.eu/europassResource/c6154165-d760-44a8-9f2d-f9fa0c306574



In [7]:
import re
import pandas as pd

# assume df has a column 'noteLiteral' with all entries (strings or None)
# 1) create a mask for URL-only entries
url_regex = re.compile(r'^\s*https?://')
is_url = df['noteLiteral'].fillna('').str.match(url_regex)

# 2) count URL vs non-URL vs null
total      = len(df)
n_url      = is_url.sum()
n_non_url  = df['noteLiteral'].notna().sum() - n_url
n_null     = df['noteLiteral'].isna().sum()

print(f"Total rows:       {total:,}")
print(f"Null entries:     {n_null:,}")
print(f"URL entries:      {n_url:,} ({n_url/total:.1%} of all rows, {n_url/(n_url+n_non_url):.1%} of non-null)")
print(f"Text entries:     {n_non_url:,} ({n_non_url/total:.1%} of all rows)")

Total rows:       26,914,461
Null entries:     24,943,362
URL entries:      1,629,696 (6.1% of all rows, 82.7% of non-null)
Text entries:     341,403 (1.3% of all rows)


##### 3. Load preprocessed json files

In [2]:
# --- Cell 1: Load & combine JSON files ---

import os, glob, json
import pandas as pd

# Your base path
BASE_PATH = "/Users/go82gax/Documents/Projekte/LFS/Analyse/Daten/QUAL Daten/json_qualifications"
os.chdir(BASE_PATH)

# Helper: safe JSON loader that returns a flat (normalized) record + filename
def load_and_flatten(filepath: str) -> pd.DataFrame:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    # If the file is a list of records, normalize the list; if it's a dict, wrap it in a list
    if isinstance(data, list):
        df = pd.json_normalize(data, sep=".")
    else:
        df = pd.json_normalize([data], sep=".")
    df.insert(0, "_source_file", os.path.basename(filepath))
    return df

# Collect all *.json files
files = sorted(glob.glob("*.json"))

# Read them all and align columns
frames = []
errors = []
for fp in files:
    try:
        frames.append(load_and_flatten(fp))
    except Exception as e:
        errors.append((fp, repr(e)))

combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print(f"Loaded {len(frames)} files into shape {combined.shape}.")
if errors:
    print(f"⚠️ {len(errors)} files failed to load. Example:", errors[0])

# Peek a few rows
display(combined.head(3))

# Optionally save an interim parquet (fast reloads later)
# combined.to_parquet("combined_qualifications.parquet", index=False)

Loaded 44486 files into shape (44486, 74).


,_source_file,uri,rdfType,identifier,title,additionalNote,homepage,supplementaryDocument,category,status,...,learningOutcomeSummary.subject.inScheme,entryRequirement.uri,entryRequirement.rdfType,entryRequirement.subject.uri,entryRequirement.subject.rdfType,entryRequirement.subject.inScheme,entryRequirement.noteLiteral,learningSetting.uri,learningSetting.prefLabel,altLabel
0,00009ee8-03da-4850-a58f-65992a9e1d98.json,http://data.europa.eu/snb/data/qualification/0...,[http://data.europa.eu/snb/model/elm/Qualifica...,[{'uri': 'http://data.europa.eu/snb/data/europ...,Master of Science in Data Analytics,[],[],[],[],released,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0001b42f-fd11-47a7-8b64-77e52a350c6a.json,http://data.europa.eu/snb/data/qualification/0...,[http://data.europa.eu/snb/model/elm/Qualifica...,[{'uri': 'http://data.europa.eu/snb/data/europ...,NaN,[{'uri': 'http://data.europa.eu/snb/data/europ...,[{'uri': 'http://data.europa.eu/snb/data/europ...,[],[],released,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0004a4c4-8ac8-482f-afea-69e500c6298c.json,http://data.europa.eu/snb/data/qualification/0...,[http://data.europa.eu/snb/model/elm/Qualifica...,[{'uri': 'http://data.europa.eu/snb/data/europ...,Master of Science in Food and Nutritional Prod...,[],[],[],[],released,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### 4. Inspect preprocessed json files

In [10]:
# --- Robust EDA + coverage for name/level/country/description (with real country names) ---

import json
import re
import numpy as np
import pandas as pd

df = combined.copy()

# ========= Helpers =========

def is_listlike(x):
    return isinstance(x, (list, tuple, set, np.ndarray))

def to_hashable(x):
    """Turn lists/dicts into hashable (and printable) forms for nunique/value_counts."""
    if isinstance(x, dict):
        # Prefer common label-like fields if present
        for k in ("prefLabel", "label", "name", "value", "id", "uri"):
            if k in x:
                return to_hashable(x[k])
        try:
            return json.dumps(x, sort_keys=True, ensure_ascii=False)
        except Exception:
            return str(x)
    if is_listlike(x):
        try:
            return tuple(to_hashable(e) for e in list(x))
        except Exception:
            return tuple(map(str, x))
    if isinstance(x, (np.generic,)):
        return x.item()
    return x

def make_hashable(df_: pd.DataFrame) -> pd.DataFrame:
    # Avoid FutureWarning about applymap: map each Series individually
    return df_.apply(lambda s: s.map(to_hashable))

def nonempty_scalar_string(x) -> bool:
    """True if x is a scalar, non-NA, and not just empty/whitespace."""
    if is_listlike(x) or isinstance(x, dict):
        return False
    return (x is not None) and (not (isinstance(x, float) and np.isnan(x))) and (str(x).strip() != "")

def coerce_text(x):
    """Coerce any object to a readable text string."""
    if isinstance(x, dict):
        for k in ("prefLabel", "label", "name", "value", "text", "title"):
            if k in x and nonempty_scalar_string(x[k]):
                return str(x[k]).strip()
        if "uri" in x and nonempty_scalar_string(x["uri"]):
            return str(x["uri"]).strip()
        return json.dumps(x, ensure_ascii=False)
    if is_listlike(x):
        if len(x) == 0:
            return ""
        # If single-item list, unwrap
        if len(x) == 1:
            return coerce_text(list(x)[0])
        # Otherwise join readable pieces
        return ", ".join(filter(None, (coerce_text(e) for e in x)))
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return str(x).strip()

def first_nonempty(*values):
    """Return first non-empty, readable text among candidates (handles lists/dicts)."""
    for v in values:
        s = coerce_text(v)
        if s.strip() != "":
            return s.strip()
    return np.nan

# ========= Country parsing: get real names, drop UUIDs =========

# ISO-2 → country label
ISO2_MAP = {
    "AT":"Austria","BE":"Belgium","BG":"Bulgaria","CH":"Switzerland","CY":"Cyprus","CZ":"Czechia",
    "DE":"Germany","DK":"Denmark","EE":"Estonia","EL":"Greece","ES":"Spain","FI":"Finland","FR":"France",
    "HR":"Croatia","HU":"Hungary","IE":"Ireland","IS":"Iceland","IT":"Italy","LI":"Liechtenstein","LT":"Lithuania",
    "LU":"Luxembourg","LV":"Latvia","MT":"Malta","NL":"Netherlands","NO":"Norway","PL":"Poland","PT":"Portugal",
    "RO":"Romania","SE":"Sweden","SI":"Slovenia","SK":"Slovakia","UK":"United Kingdom"
}
UUID_RE = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.I)
ISO2_RE = re.compile(r'\b([A-Z]{2})\b')

def last_path_segment(s: str) -> str:
    seg = s.rstrip("/").split("/")[-1]
    seg = seg.split("#")[-1].split("?")[0]
    return seg

def normalize_token_to_country(token: str) -> str:
    """Return a country name if token looks like ISO-2 or a readable country word; drop UUID-like."""
    t = token.strip()
    if not t or UUID_RE.match(t):
        return ""  # drop UUIDs / empties
    # pure ISO-2
    if len(t) == 2 and t.upper() in ISO2_MAP:
        return ISO2_MAP[t.upper()]
    # sometimes tokens embed ISO-2 (e.g., 'country/DE' already stripped to 'DE')
    # otherwise, keep as-is if it looks like a word (will be human-readable)
    return t

def find_country_signals(obj):
    """
    Recursively search objects/lists for country-ish fields:
    - prefLabel/label/name/value/text/title (human label)
    - fields named 'country', 'addressCountry' or 'countryCode'
    - URIs containing '/country/' or ending with '/[A-Z]{2}'
    - ISO-2 codes embedded in text (last resort)
    """
    out = []

    if isinstance(obj, dict):
        # direct country-ish keys first
        for k in ("country", "addressCountry", "countryCode"):
            if k in obj and nonempty_scalar_string(obj[k]):
                out.append(str(obj[k]).strip())
        # label-like keys
        for k in ("prefLabel", "label", "name", "value", "text", "title"):
            if k in obj and nonempty_scalar_string(obj[k]):
                out.append(str(obj[k]).strip())
        # URIs that may carry country info
        if "uri" in obj and nonempty_scalar_string(obj["uri"]):
            u = str(obj["uri"]).strip()
            if "/country/" in u or "/countries/" in u:
                out.append(last_path_segment(u))
            else:
                # If last segment is ISO-2, use it; ignore UUIDs
                out.append(last_path_segment(u))
        # Recurse remaining fields
        for v in obj.values():
            out.extend(find_country_signals(v))

    elif is_listlike(obj):
        for e in obj:
            out.extend(find_country_signals(e))

    elif nonempty_scalar_string(obj):
        s = str(obj).strip()
        # URI case
        if s.startswith("http://") or s.startswith("https://"):
            out.append(last_path_segment(s))
        else:
            # look for standalone ISO-2 in free text (e.g., "... (PL)")
            for m in ISO2_RE.findall(s):
                if m.upper() in ISO2_MAP:
                    out.append(m.upper())
            out.append(s)

    # Normalize & clean
    cleaned = []
    for t in out:
        name = normalize_token_to_country(t)
        if name:
            cleaned.append(name)
    # de-dup, preserve order
    seen = set()
    uniq = []
    for c in cleaned:
        if c not in seen:
            uniq.append(c)
            seen.add(c)
    return uniq

def extract_country_from_any(x):
    tokens = find_country_signals(x)
    return ", ".join(tokens)

def pct_fmt(count, total):
    return 0.0 if total == 0 else 100.0 * count / total

# ========= EDA bits (cardinality etc.) with hashable view =========

df_hashable = make_hashable(df)

card = df_hashable.nunique(dropna=True).sort_values(ascending=False)
print("Cardinality (unique values per column):")
display(card.to_frame("unique_values"))

# ========= Target fields & coverage =========
# NOTE: country candidates now prioritize 'awardingOpportunity' before 'publisher.location'
name_candidates   = [c for c in ["title", "altLabel"] if c in df.columns]
level_candidates  = [c for c in ["EQFLevel.prefLabel", "NQFLevel", "ISCEDFCode"] if c in df.columns]
country_candidates = [c for c in ["awardingOpportunity", "publisher.location", "publisher.legalName"] if c in df.columns]
desc_candidates   = [c for c in ["description", "learningOutcomeSummary.noteLiteral", "additionalNote"] if c in df.columns]

print("\nCandidates used ->")
print("  name:", name_candidates)
print("  level:", level_candidates)
print("  country:", country_candidates, "(awardingOpportunity prioritized)")
print("  description:", desc_candidates)

# Derived columns (robust against lists/dicts/URIs)
df["_name"]  = df.apply(lambda r: first_nonempty(*[r[c] for c in name_candidates]) if name_candidates else np.nan, axis=1)
df["_level"] = df.apply(lambda r: first_nonempty(*[r[c] for c in level_candidates]) if level_candidates else np.nan, axis=1)

# Country: try candidates, then parse to readable label(s)
raw_country = df.apply(lambda r: first_nonempty(*[r[c] for c in country_candidates]) if country_candidates else np.nan, axis=1)
df["_country"] = raw_country.apply(extract_country_from_any)

# Description: prefer 'description' then 'learningOutcomeSummary.noteLiteral' then 'additionalNote'
df["_desc"]  = df.apply(lambda r: first_nonempty(*[r[c] for c in desc_candidates]) if desc_candidates else np.nan, axis=1)

# Coverage stats
N = len(df)
coverage = {
    "name":        int(df["_name"].astype(str).str.strip().ne("").sum()),
    "level":       int(df["_level"].astype(str).str.strip().ne("").sum()),
    "country":     int(df["_country"].astype(str).str.strip().ne("").sum()),
    "description": int(df["_desc"].astype(str).str.strip().ne("").sum()),
}
coverage_df = (
    pd.DataFrame([
        {"field": k, "non_empty_rows": v, "percent": round(pct_fmt(v, N), 2)}
        for k, v in coverage.items()
    ])
    .set_index("field")
    .sort_index()
)

print(f"\nTotal rows: {N:,}")
print("\nCoverage for required fields (rows & %):")
display(coverage_df)

# Quick top values to sanity-check parsing
print("\nTop countries (parsed to names/labels):")
display(df["_country"].value_counts(dropna=True).head(30).to_frame("rows"))

print("\nTop level values:")
display(pd.Series(df["_level"]).value_counts(dropna=True).head(30).to_frame("rows"))

# Description length stats (how "short" and presence)
desc_lens = df["_desc"].fillna("").astype(str).str.len()
print("\nDescription length stats (characters):")
display(desc_lens.describe(percentiles=[.01,.05,.25,.5,.75,.9,.95,.99]).to_frame("char_count").T)

# Show a few good examples
print("\nSample rows with all four fields present:")
mask_all = (
    df["_name"].astype(str).str.strip().ne("") &
    df["_level"].astype(str).str.strip().ne("") &
    df["_country"].astype(str).str.strip().ne("") &
    df["_desc"].astype(str).str.strip().ne("")
)
display(df.loc[mask_all, ["_name", "_level", "_country", "_desc", "_source_file"]].head(10))

# (Optional) essentials export for downstream work:
# essentials = df[["_name", "_level", "_country", "_desc", "_source_file"]]
# essentials.to_parquet("qualifications_essentials.parquet", index=False)
# essentials.to_csv("qualifications_essentials.csv", index=False)


Cardinality (unique values per column):


,unique_values
_source_file,44486
identifier,44486
uri,44486
learningOutcome,43354
awardingOpportunity,34347
...,...
publisher.contactPoint,1
publisher.hasSubOrganization,1
mode,1
publisher.status,1



Candidates used ->
  name: ['title', 'altLabel']
  level: ['EQFLevel.prefLabel', 'NQFLevel', 'ISCEDFCode']
  country: ['awardingOpportunity', 'publisher.location', 'publisher.legalName'] (awardingOpportunity prioritized)
  description: ['description', 'learningOutcomeSummary.noteLiteral', 'additionalNote']

Total rows: 44,486

Coverage for required fields (rows & %):


,non_empty_rows,percent
field,,
country,0,0.0
description,44486,100.0
level,44486,100.0
name,44486,100.0



Top countries (parsed to names/labels):


...

,rows
_level,
Level 7,9900
Level 5,9900
Level 6,9900
Level 4,8840
Level 3,3686
Level 8,1315
Level 2,738
Level 1,207



Description length stats (characters):


,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,max
char_count,44486.0,295.685362,684.652336,0.0,0.0,0.0,0.0,0.0,342.0,932.0,1311.0,3048.0,20532.0



Sample rows with all four fields present:


,_name,_level,_country,_desc,_source_file


In [11]:
# --- Quick inspection tables from derived fields ---

import numpy as np
import pandas as pd

def nonempty(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip()
    return (s != "") & (s.str.lower() != "nan")

N = len(df)

# Coverage
coverage_df = pd.DataFrame({
    "field": ["name", "level", "country", "description_any", "description_20chars"],
    "non_empty_rows": [
        int(nonempty(df["_name"]).sum()),
        int(nonempty(df["_level"]).sum()),
        int(nonempty(df["_country"]).sum()),
        int((df["_desc"].fillna("").astype(str).str.len() >= 1).sum()),
        int((df["_desc"].fillna("").astype(str).str.len() >= 20).sum()),
    ],
})
coverage_df["percent"] = (coverage_df["non_empty_rows"] / N * 100).round(2)

# EQF level distribution
eqf_dist = (
    df["_level"].value_counts(dropna=False)
      .rename_axis("level")
      .to_frame("rows")
      .assign(percent=lambda x: (x["rows"]/N*100).round(2))
      .reset_index()
)

# Country distribution (top 30)
country_dist = (
    df["_country"].value_counts(dropna=False)
      .head(30)
      .rename_axis("country")
      .to_frame("rows")
      .assign(percent=lambda x: (x["rows"]/N*100).round(2))
      .reset_index()
)

# Description length buckets
bins   = [-0.1, 0.5, 20, 50, 100, 250, 500, 1000, 5000, np.inf]
labels = ["0", "1–19", "20–49", "50–99", "100–249", "250–499", "500–999", "1000–4999", "5000+"]
desc_len = df["_desc"].fillna("").astype(str).str.len()
desc_bucket = pd.cut(desc_len, bins=bins, labels=labels, right=False, include_lowest=True)
desc_len_dist = (
    desc_bucket.value_counts().sort_index().to_frame("rows")
    .assign(percent=lambda x: (x["rows"]/N*100).round(2))
    .reset_index().rename(columns={"index":"bucket"})
)

# Curated examples
examples_good = df.loc[
    nonempty(df["_name"]) &
    nonempty(df["_level"]) &
    nonempty(df["_country"]) &
    (df["_desc"].fillna("").astype(str).str.len() >= 80),
    ["_name", "_level", "_country", "_desc", "_source_file"]
].head(15)

examples_weak_desc = df.loc[
    (df["_desc"].fillna("").astype(str).str.len() < 20),
    ["_name", "_level", "_country", "_desc", "_source_file"]
].head(15)

# Show
print(f"Total rows: {N:,}")
print("\nCoverage:")
display(coverage_df)

print("\nEQF level distribution:")
display(eqf_dist)

print("\nTop 30 countries:")
display(country_dist)

print("\nDescription length buckets:")
display(desc_len_dist)

print("\nExamples — good (all present, strong description):")
display(examples_good)

print("\nExamples — weak descriptions (<20 chars after cleaning):")
display(examples_weak_desc)

# Optional: export
# coverage_df.to_csv("coverage_strict.csv", index=False)
# eqf_dist.to_csv("eqf_distribution.csv", index=False)
# country_dist.to_csv("top_country_tokens.csv", index=False)
# desc_len_dist.to_csv("description_length_buckets.csv", index=False)
# examples_good.to_csv("examples_good.csv", index=False)
# examples_weak_desc.to_csv("examples_weak_desc.csv", index=False)


Total rows: 44,486

Coverage:


,field,non_empty_rows,percent
0,name,28464,63.98
1,level,44486,100.00
2,country,0,0.00
3,description_any,19535,43.91
4,description_20chars,19519,43.88



EQF level distribution:


,level,rows,percent
0,Level 7,9900,22.25
1,Level 5,9900,22.25
2,Level 6,9900,22.25
3,Level 4,8840,19.87
4,Level 3,3686,8.29
5,Level 8,1315,2.96
6,Level 2,738,1.66
7,Level 1,207,0.47



Top 30 countries:


...

,_desc,rows,percent
0,0,24951,56.09
1,1–19,16,0.04
2,20–49,47,0.11
3,50–99,1270,2.85
4,100–249,5107,11.48
5,250–499,5387,12.11
6,500–999,4270,9.60
7,1000–4999,3322,7.47
8,5000+,116,0.26



Examples — good (all present, strong description):


,_name,_level,_country,_desc,_source_file



Examples — weak descriptions (<20 chars after cleaning):


,_name,_level,_country,_desc,_source_file
0,Master of Science in Data Analytics,Level 7,,NaN,00009ee8-03da-4850-a58f-65992a9e1d98.json
2,Master of Science in Food and Nutritional Prod...,Level 7,,NaN,0004a4c4-8ac8-482f-afea-69e500c6298c.json
4,NaN,Level 4,,NaN,000a8c30-35cc-4408-ae15-662e5b3da0e4.json
5,Bachelor of Science in Computing,Level 6,,NaN,000ed40e-4cb2-4d1a-ab95-48958f9dcc12.json
6,MSc,Level 7,,NaN,0011f24a-eee3-4a66-8950-a61de36696d9.json
11,NaN,Level 4,,NaN,0023deff-f4e3-43a6-82a1-8978964d00ff.json
12,NaN,Level 3,,NaN,00254d1f-9f3b-4e4f-9fe3-a2161daed91f.json
14,Postgraduate Diploma in Artificial Intelligenc...,Level 7,,NaN,002a50d7-66f0-4076-9077-ce0fc02f9f8c.json
16,Bachelor of Arts,Level 6,,NaN,00313489-1aec-4430-9143-33801cab55bc.json
17,BSc in Nursing (Mental Health),Level 6,,NaN,00351d41-4ffa-41a3-8d52-5d36506bd70e.json


In [12]:
# --- Cell A: Diagnose country-ish fields ---

import re
import pandas as pd
import numpy as np
from urllib.parse import urlparse

df = combined.copy()

def to_list(x):
    if isinstance(x, list): return x
    if isinstance(x, tuple): return list(x)
    if x is None or (isinstance(x, float) and np.isnan(x)): return []
    return [x]

def deep_values(obj, keys=("prefLabel","label","name","value","text","title","uri","country","addressCountry","countryCode")):
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in keys:
                out.append(v)
            out.extend(deep_values(v, keys=keys))
    elif isinstance(obj, (list, tuple)):
        for e in obj:
            out.extend(deep_values(e, keys=keys))
    return out

def last_seg(s):
    s = str(s).strip()
    if not s: return ""
    s = s.rstrip("/").split("/")[-1]
    s = s.split("#")[-1].split("?")[0]
    return s

UUID_RE = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.I)
ISO2_MAP = {"AT":"Austria","BE":"Belgium","BG":"Bulgaria","CH":"Switzerland","CY":"Cyprus","CZ":"Czechia",
            "DE":"Germany","DK":"Denmark","EE":"Estonia","EL":"Greece","ES":"Spain","FI":"Finland","FR":"France",
            "HR":"Croatia","HU":"Hungary","IE":"Ireland","IS":"Iceland","IT":"Italy","LI":"Liechtenstein","LT":"Lithuania",
            "LU":"Luxembourg","LV":"Latvia","MT":"Malta","NL":"Netherlands","NO":"Norway","PL":"Poland","PT":"Portugal",
            "RO":"Romania","SE":"Sweden","SI":"Slovenia","SK":"Slovakia","UK":"United Kingdom"}

def iso2_or_empty(token):
    t = token.strip().upper()
    return ISO2_MAP.get(t, "")

country_fields = [c for c in ["awardingOpportunity","publisher.location","publisher.legalName"] if c in df.columns]

# 1) What keys/types appear inside these fields?
def struct_signature(x):
    if isinstance(x, dict): return "dict:" + ",".join(sorted(set(x.keys())))
    if isinstance(x, list):
        inner = [struct_signature(e) for e in x[:3]]
        return "list[" + "|".join(inner) + ("|..." if len(x)>3 else "") + "]"
    return type(x).__name__

for col in country_fields:
    sigs = df[col].dropna().head(200).map(struct_signature).value_counts().head(10)
    print(f"\n{col} — common structures (sample of 200 non-nulls):")
    print(sigs)

# 2) Collect URIs (if any) and analyze last segments/domains
def all_uris(col):
    uris = []
    for val in df[col].dropna():
        for v in to_list(val):
            if isinstance(v, dict) and "uri" in v:
                uris.append(str(v["uri"]))
            elif isinstance(v, str) and v.startswith(("http://","https://")):
                uris.append(v)
    return uris

for col in ["awardingOpportunity","publisher.location"]:
    if col in df.columns:
        uris = all_uris(col)
        print(f"\n{col}: found {len(uris):,} URIs (not unique)")
        # last segments
        segs = pd.Series([last_seg(u) for u in uris if u]).value_counts().head(20)
        print("Top last segments (could be ISO-2 or UUIDs):")
        print(segs)
        # domains
        domains = pd.Series([urlparse(u).netloc for u in uris if u]).value_counts().head(10)



awardingOpportunity — common structures (sample of 200 non-nulls):
awardingOpportunity
list[dict:awardingBody,identifier,learningAchievementSpecification,rdfType,uri]             103
list[dict:awardingBody,identifier,learningAchievementSpecification,location,rdfType,uri]     49
list[]                                                                                       48
Name: count, dtype: int64

publisher.location — common structures (sample of 200 non-nulls):
publisher.location
list[dict:address,geometry,identifier,rdfType,spatialCode,uri]    200
Name: count, dtype: int64

publisher.legalName — common structures (sample of 200 non-nulls):
publisher.legalName
str    200
Name: count, dtype: int64

awardingOpportunity: found 34,346 URIs (not unique)
Top last segments (could be ISO-2 or UUIDs):
b70abc21-bff7-4343-8634-8ddceaeea9d7    1
69817bc1-bb2f-4739-af5d-7e5cadbc91e1    1
951b4b88-161a-4034-8dc8-7f4bd9cd4e23    1
90abfd72-aed6-4c31-9885-aea6e20e1463    1
3443fa70-cda9-4cf0-8c84-b

In [13]:
# --- Cell 1: Clean descriptions, keep only rows with useful text, save subset ---

import re, json
import numpy as np
import pandas as pd
from html import unescape

df = combined.copy()

# --- helpers ---
def is_listlike(x):
    return isinstance(x, (list, tuple, set, np.ndarray))

def coerce_text(x):
    if isinstance(x, dict):
        for k in ("prefLabel", "label", "name", "value", "text", "title", "noteLiteral"):
            if k in x and x[k] not in (None, "", []):
                return str(x[k]).strip()
        if "uri" in x and x["uri"] not in (None, "", []):
            return str(x["uri"]).strip()
        return json.dumps(x, ensure_ascii=False)
    if is_listlike(x):
        if len(x) == 0: return ""
        if len(x) == 1: return coerce_text(list(x)[0])
        return ", ".join(filter(None, (coerce_text(e) for e in x)))
    if x is None or (isinstance(x, float) and np.isnan(x)): return ""
    return str(x).strip()

URL_RE = re.compile(r'^https?://', re.I)
TAG_RE = re.compile(r'<[^>]+>')
WS_RE  = re.compile(r'\s+')

def clean_description(x: str) -> str:
    s = coerce_text(x)
    if not s: return ""
    if URL_RE.match(s):
        return ""  # treat raw URLs as not-a-description
    s = unescape(s)
    s = TAG_RE.sub(' ', s)
    s = WS_RE.sub(' ', s).strip()
    return s

def first_nonempty(*vals):
    for v in vals:
        s = coerce_text(v)
        if s.strip():
            return s.strip()
    return ""

# --- choose description candidates (in priority order) ---
desc_candidates = [c for c in ["description", "learningOutcomeSummary.noteLiteral", "additionalNote"] if c in df.columns]

# build cleaned description
df["_desc_raw"] = df.apply(lambda r: first_nonempty(*[r[c] for c in desc_candidates]) if desc_candidates else "", axis=1)
df["_desc"]     = df["_desc_raw"].map(clean_description)

# define “has a useful description” (>= 20 characters after cleaning)
has_desc = df["_desc"].str.len().fillna(0) >= 20
df_desc = df.loc[has_desc].copy()

print(f"Kept {len(df_desc):,} / {len(df):,} rows with cleaned description (>=20 chars).")

# (optional) also derive name/level for later use
name_candidates  = [c for c in ["title", "altLabel"] if c in df.columns]
level_candidates = [c for c in ["EQFLevel.prefLabel", "NQFLevel", "ISCEDFCode"] if c in df.columns]
df_desc["_name"]  = df_desc.apply(lambda r: first_nonempty(*[r[c] for c in name_candidates]) if name_candidates else "", axis=1)
df_desc["_level"] = df_desc.apply(lambda r: first_nonempty(*[r[c] for c in level_candidates]) if level_candidates else "", axis=1)

# save an interim file (change path if you like)
df_desc.to_parquet("qualifications_with_descriptions.parquet", index=False)
df_desc.to_csv("qualifications_with_descriptions.csv", index=False)
print("Saved: qualifications_with_descriptions.parquet & .csv")


Kept 17,200 / 44,486 rows with cleaned description (>=20 chars).
Saved: qualifications_with_descriptions.parquet & .csv


In [14]:
# --- Cell 2: Heuristic country backfill into _country_final ---

import re
from urllib.parse import urlparse

df_desc = df_desc.copy()

ISO2_MAP = {
    "AT":"Austria","BE":"Belgium","BG":"Bulgaria","CH":"Switzerland","CY":"Cyprus","CZ":"Czechia",
    "DE":"Germany","DK":"Denmark","EE":"Estonia","EL":"Greece","ES":"Spain","FI":"Finland","FR":"France",
    "HR":"Croatia","HU":"Hungary","IE":"Ireland","IS":"Iceland","IT":"Italy","LI":"Liechtenstein","LT":"Lithuania",
    "LU":"Luxembourg","LV":"Latvia","MT":"Malta","NL":"Netherlands","NO":"Norway","PL":"Poland","PT":"Portugal",
    "RO":"Romania","SE":"Sweden","SI":"Slovenia","SK":"Slovakia","UK":"United Kingdom"
}
TLD_TO_COUNTRY = {
    "at":"Austria","be":"Belgium","bg":"Bulgaria","ch":"Switzerland","cy":"Cyprus","cz":"Czechia",
    "de":"Germany","dk":"Denmark","ee":"Estonia","gr":"Greece","es":"Spain","fi":"Finland","fr":"France",
    "hr":"Croatia","hu":"Hungary","ie":"Ireland","is":"Iceland","it":"Italy","li":"Liechtenstein","lt":"Lithuania",
    "lu":"Luxembourg","lv":"Latvia","mt":"Malta","nl":"Netherlands","no":"Norway","pl":"Poland","pt":"Portugal",
    "ro":"Romania","se":"Sweden","si":"Slovenia","sk":"Slovakia","uk":"United Kingdom"
}
ISO2_RE = re.compile(r'\b([A-Z]{2})\b')
UUID_RE = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.I)

def to_list(x):
    if isinstance(x, list): return x
    if isinstance(x, tuple): return list(x)
    if x is None or (isinstance(x, float) and np.isnan(x)): return []
    return [x]

def deep_collect(obj):
    """Collect strings (labels/uris) from nested awarding/location structures."""
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in ("prefLabel","label","name","value","text","title","uri","country","addressCountry","countryCode","spatialCode","homepage"):
                out.append(v)
            out.extend(deep_collect(v))
    elif isinstance(obj, (list, tuple)):
        for e in obj:
            out.extend(deep_collect(e))
    else:
        out.append(obj)
    return out

def last_seg(s):
    s = str(s).strip()
    if not s: return ""
    s = s.rstrip("/").split("/")[-1]
    s = s.split("#")[-1].split("?")[0]
    return s

def guess_from_iso2(val):
    tokens = deep_collect(val)
    for t in tokens:
        if not isinstance(t, str):
            continue
        # pull ISO-2 from anywhere in the string (URI segments, codes, text)
        for code in ISO2_RE.findall(t):
            code = code.upper()
            if code in ISO2_MAP:
                return ISO2_MAP[code]
        # also try last segment if it looks like a two-letter code
        if t.startswith(("http://","https://")):
            seg = last_seg(t)
            if seg.upper() in ISO2_MAP:
                return ISO2_MAP[seg.upper()]
    return ""

def guess_from_homepage(url):
    # Try 'publisher.homepage' or top-level 'homepage'
    if not isinstance(url, str) or not url.strip():
        return ""
    try:
        host = urlparse(url).hostname or ""
        if not host:
            return ""
        # take the last TLD label; e.g., 'ac.uk' -> 'uk'
        tld = host.split(".")[-1].lower()
        return TLD_TO_COUNTRY.get(tld, "")
    except Exception:
        return ""

# pull candidate columns if present
awarding = df_desc.get("awardingOpportunity")
ploc     = df_desc.get("publisher.location")
phome    = df_desc.get("publisher.homepage", df_desc.get("homepage", pd.Series([""]*len(df_desc))))

iso_guess_awarding = awarding.map(guess_from_iso2) if awarding is not None else pd.Series([""]*len(df_desc))
iso_guess_ploc     = ploc.map(guess_from_iso2)     if ploc is not None     else pd.Series([""]*len(df_desc))
tld_guess          = phome.map(guess_from_homepage)

def first_nonempty(*vals):
    for v in vals:
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

df_desc["_country_final"] = [
    first_nonempty(a, p, t)
    for a, p, t in zip(iso_guess_awarding.fillna(""), iso_guess_ploc.fillna(""), tld_guess.fillna(""))
]

print("Heuristic country coverage on the subset:",
      f"{(df_desc['_country_final'].astype(str).str.strip().ne('').mean()*100):.2f}% non-empty")


Heuristic country coverage on the subset: 0.45% non-empty


In [15]:
# --- Cell 3: Descriptives on the subset + tidy export ---

import numpy as np
import pandas as pd

N = len(df_desc)
print(f"Subset size: {N:,} rows")

# Basic coverage in subset
coverage = pd.DataFrame({
    "field": ["name", "level", "country_final", "desc_>=20chars", "desc_>=80chars"],
    "non_empty_rows": [
        int(df_desc["_name"].fillna("").astype(str).str.strip().ne("").sum()),
        int(df_desc["_level"].fillna("").astype(str).str.strip().ne("").sum()),
        int(df_desc["_country_final"].fillna("").astype(str).str.strip().ne("").sum()),
        int((df_desc["_desc"].str.len() >= 20).sum()),
        int((df_desc["_desc"].str.len() >= 80).sum()),
    ],
})
coverage["percent"] = (coverage["non_empty_rows"]/N*100).round(2)
print("\nCoverage in the description subset:")
display(coverage)

# Description length stats & buckets
desc_len = df_desc["_desc"].str.len().fillna(0)
stats = desc_len.describe(percentiles=[.01,.05,.25,.5,.75,.9,.95,.99]).to_frame("char_count")
print("\nDescription length stats (chars):")
display(stats.T)

bins   = [-0.1, 80, 150, 300, 600, 1200, np.inf]
labels = ["<80", "80–149", "150–299", "300–599", "600–1199", "1200+"]
bucket = pd.cut(desc_len, bins=bins, labels=labels, right=False, include_lowest=True)
dist = bucket.value_counts().sort_index().to_frame("rows")
dist["percent"] = (dist["rows"]/N*100).round(2)
print("\nDescription length buckets:")
display(dist)

# Country distribution (top 25) and EQF in subset
print("\nTop countries (heuristic) — subset:")
display(df_desc["_country_final"].value_counts().head(25).to_frame("rows"))

print("\nEQF/Level distribution — subset:")
display(df_desc["_level"].value_counts().to_frame("rows"))

# Curated examples
print("\nExamples — strong descriptions (>= 150 chars):")
examples_strong = df_desc.loc[df_desc["_desc"].str.len() >= 150, ["_name","_level","_country_final","_desc","_source_file"]].head(10)
display(examples_strong)

print("\nExamples — shorter but present (20–79 chars):")
examples_short = df_desc.loc[(df_desc["_desc"].str.len().between(20,79)), ["_name","_level","_country_final","_desc","_source_file"]].head(10)
display(examples_short)

# Tidy export for downstream (essentials only)
essentials = df_desc[["_name","_level","_country_final","_desc","_source_file"]].rename(
    columns={"_country_final":"_country"}
)
essentials.to_parquet("qualifications_essentials_with_desc.parquet", index=False)
essentials.to_csv("qualifications_essentials_with_desc.csv", index=False)
print("\nSaved: qualifications_essentials_with_desc.parquet & .csv")


Subset size: 17,200 rows

Coverage in the description subset:


,field,non_empty_rows,percent
0,name,13302,77.34
1,level,17200,100.00
2,country_final,78,0.45
3,desc_>=20chars,17200,100.00
4,desc_>=80chars,17156,99.74



Description length stats (chars):


,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,max
char_count,17200.0,718.702965,851.800519,40.0,141.0,180.0,195.0,413.0,932.0,1513.0,2167.15,4040.26,19770.0



Description length buckets:


...


Examples — strong descriptions (>= 150 chars):


,_name,_level,_country_final,_desc,_source_file
3,Bachelor's degree in theological studies and ...,Level 6,,Students will be able to: demonstrate autonomo...,000823cf-00b5-44f7-afbe-353a72b2e17d.json
7,,Level 5,,National Qualification Framework (NQF): This q...,0015d0d8-d50a-41c8-aeed-254d2e315c3b.json
8,Roma coordinator,Level 4,,The candidate is able to: plan and organize th...,001abebf-e1fe-4f21-87dd-4a1c30d9cef1.json
9,41-045-H Pig breeder and keeper,Level 3,,http://www.msmt.cz/areas-of-work/further-educa...,001ac8fa-5ab8-46b1-b450-806be7b67f76.json
10,,Level 6,,National Qualification Framework (NQF): This q...,001d7bc1-54a7-416d-82c0-1844cb8a85c6.json
13,Investment fund specialist (m/f),Level 4,,This qualification is a dual vocational educat...,0028f3ae-613d-48e3-ad39-b1368f632347.json
15,Basket maker,Level 4,,"Candidates will be able to: plan, prepare and ...",002cf865-3f9b-4a06-9be1-96441d4bae14.json
18,,Level 4,,National Qualification Framework (NQF): This q...,003603bc-9099-4486-aaf4-cf1ed6dac0a9.json
24,,Level 6,,National Qualification Framework (NQF): This q...,003cf125-c2dd-4511-ac1c-33edb3a3148c.json
26,,Level 5,,National Qualification Framework (NQF): This q...,00483ee2-7f82-48a1-94b6-f3aa536c5002.json



Examples — shorter but present (20–79 chars):


,_name,_level,_country_final,_desc,_source_file
599,Diploma in Food and Beverage Service Operations,Level 4,,Typical programme duration is 2 Semesters + II...,03643875-ba22-4890-9fcc-8a72aed3ab14.json
2177,Office Skills Award,Level 4,,Contact provider for further information,0c8b2b7f-81f1-4b4a-8b8b-27a046a37139.json
2543,Award in Information Technology,Level 2,,The aim of this module is to provide basic IT ...,0ed990b8-ec20-47ec-be3d-89b34a19f757.json
6085,The Applied Insurance Certificate (Continuous ...,Level 4,,Contact provider for further information,23b4b64e-7645-4475-acc5-18640070e8ff.json
6646,Diploma in Food preparation & Productions Oper...,Level 4,,"2 Semesters IITP (1 Year), Typical programme d...",27244619-6633-4d4f-85ad-9e67aee9d9ad.json
7902,The Advanced Applied Insurance Certificate (Co...,Level 4,,Contact provider for further information,2e68454d-c9c3-41cb-b6f3-e1a285b2ad9e.json
11222,Certified Privacy Practitioner/ EU-DPO (CPP/EU...,Level 5,,Contact provider for further information,40ecd93e-8894-43f9-8187-1916f20ef7d0.json
11449,Level 4 Advanced Diploma in Water Technology a...,Level 4,,Contact provider for further information,425b68de-227d-4f4b-be50-8adabbb4ade6.json
11651,Human Resources Competency Certificate,Level 4,,Contact provider for further information,436a06fd-4e38-4b79-8967-689d5862b3ed.json
13881,Undergraduate Certificate in Sportsbook,Level 5,,Target Group: Those aiming for a career in Spo...,4fd12067-5bf2-4c56-a7c4-69e064626c8b.json



Saved: qualifications_essentials_with_desc.parquet & .csv


In [16]:
# --- Filter out boilerplate/links from description, then re-run descriptives & export ---

import re
import numpy as np
import pandas as pd

# Assumes df_desc already exists from previous step and includes columns: _name, _level, _country_final, _desc, _source_file
# If you named it differently, swap df_desc below to your variable.

df_desc = df_desc.copy()

# 1) Define filters
LINK_RE = re.compile(r'(https?://\S+|www\.\S+)', flags=re.IGNORECASE)

# We match both the long phrase and short forms/variants of NQF
PHRASES_BLOCK = [
    r'\bcontact\s+provider\s+for\s+further\s+information\b',
    r'\bnational\s+qualification\s+framework\s*\(?\s*nqf\s*\)?\b',
    r'\bNQF\b',  # keep this to catch short mentions
]
PHRASES_RE = re.compile("|".join(PHRASES_BLOCK), flags=re.IGNORECASE)

def is_boilerplate_or_link(text: str) -> bool:
    if not isinstance(text, str):
        return False
    s = text.strip()
    if not s:
        return False
    if LINK_RE.search(s):
        return True
    if PHRASES_RE.search(s):
        return True
    return False

mask_bad = df_desc["_desc"].apply(is_boilerplate_or_link)
removed = int(mask_bad.sum())

df_desc_clean = df_desc.loc[~mask_bad].copy()
kept = len(df_desc_clean)
orig = len(df_desc)

print(f"Filtered out {removed:,} of {orig:,} rows ({removed/orig*100:.2f}%). Kept {kept:,} rows.")

# 2) Descriptives on the cleaned subset
N = len(df_desc_clean)

def nonempty(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip()
    return (s != "") & (s.str.lower() != "nan")

coverage = pd.DataFrame({
    "field": ["name", "level", "country_final", "desc_>=20chars", "desc_>=80chars"],
    "non_empty_rows": [
        int(nonempty(df_desc_clean["_name"]).sum()),
        int(nonempty(df_desc_clean["_level"]).sum()),
        int(nonempty(df_desc_clean["_country_final"]).sum()),
        int((df_desc_clean["_desc"].str.len() >= 20).sum()),
        int((df_desc_clean["_desc"].str.len() >= 80).sum()),
    ],
})
coverage["percent"] = (coverage["non_empty_rows"]/N*100).round(2)

print("\nCoverage in the CLEANED description subset:")
display(coverage)

desc_len = df_desc_clean["_desc"].str.len().fillna(0)
stats = desc_len.describe(percentiles=[.01,.05,.25,.5,.75,.9,.95,.99]).to_frame("char_count")
print("\nDescription length stats (chars) — CLEANED subset:")
display(stats.T)

bins   = [-0.1, 80, 150, 300, 600, 1200, np.inf]
labels = ["<80", "80–149", "150–299", "300–599", "600–1199", "1200+"]
bucket = pd.cut(desc_len, bins=bins, labels=labels, right=False, include_lowest=True)
dist = bucket.value_counts().sort_index().to_frame("rows")
dist["percent"] = (dist["rows"]/N*100).round(2)
print("\nDescription length buckets — CLEANED subset:")
display(dist)

print("\nTop countries (heuristic) — CLEANED subset:")
display(df_desc_clean["_country_final"].value_counts().head(25).to_frame("rows"))

print("\nEQF/Level distribution — CLEANED subset:")
display(df_desc_clean["_level"].value_counts().to_frame("rows"))

print("\nExamples — strong descriptions (>= 150 chars) — CLEANED subset:")
examples_strong = df_desc_clean.loc[df_desc_clean["_desc"].str.len() >= 150, ["_name","_level","_country_final","_desc","_source_file"]].head(10)
display(examples_strong)

print("\nExamples — shorter but present (20–79 chars) — CLEANED subset:")
examples_short = df_desc_clean.loc[(df_desc_clean["_desc"].str.len().between(20,79)), ["_name","_level","_country_final","_desc","_source_file"]].head(10)
display(examples_short)

# 3) Export a slim essentials file for downstream use
essentials_clean = df_desc_clean[["_name","_level","_country_final","_desc","_source_file"]].rename(
    columns={"_country_final":"_country"}
)
essentials_clean.to_parquet("qualifications_essentials_with_desc_clean.parquet", index=False)
essentials_clean.to_csv("qualifications_essentials_with_desc_clean.csv", index=False)
print("\nSaved: qualifications_essentials_with_desc_clean.parquet & .csv")


Filtered out 6,986 of 17,200 rows (40.62%). Kept 10,214 rows.

Coverage in the CLEANED description subset:


,field,non_empty_rows,percent
0,name,10212,99.98
1,level,10214,100.00
2,country_final,0,0.00
3,desc_>=20chars,10214,100.00
4,desc_>=80chars,10189,99.76



Description length stats (chars) — CLEANED subset:


,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,max
char_count,10214.0,1003.601527,982.678838,49.0,120.13,220.65,473.0,806.0,1063.0,1951.7,2688.0,4836.09,19770.0



Description length buckets — CLEANED subset:


...


Examples — strong descriptions (>= 150 chars) — CLEANED subset:


,_name,_level,_country_final,_desc,_source_file
3,Bachelor's degree in theological studies and ...,Level 6,,Students will be able to: demonstrate autonomo...,000823cf-00b5-44f7-afbe-353a72b2e17d.json
8,Roma coordinator,Level 4,,The candidate is able to: plan and organize th...,001abebf-e1fe-4f21-87dd-4a1c30d9cef1.json
13,Investment fund specialist (m/f),Level 4,,This qualification is a dual vocational educat...,0028f3ae-613d-48e3-ad39-b1368f632347.json
15,Basket maker,Level 4,,"Candidates will be able to: plan, prepare and ...",002cf865-3f9b-4a06-9be1-96441d4bae14.json
28,"Award in Politics, Power and the State",Level 6,,This module explores the theories on Politics ...,004a2bf5-e006-446f-86b4-e2fe33169af2.json
30,ELECTRONICS AND AUTOMATION TECHNICIAN,Level 4,,The candidate is able to: Electronics and Auto...,004be339-d304-4bdf-ac58-f3c6bb3f7dbe.json
31,"Skilled floristry, green retail trade and styl...",Level 3,,"The Skilled floristry, green retail trade and ...",004cbe98-8027-4f5c-8135-32165d1a42d0.json
41,Master of Business Administration (MBA) in Blo...,Level 7,,This course provides an up-to-date overview of...,0057ef44-5673-40a1-8b94-a6f67f97fb32.json
44,Peritoneal Dialysis Extramural,Level 6,,"Due to the large number of PD clients, this ca...",005ab4c5-e073-46f9-b022-03db44c86f48.json
55,Award in English Language for Foreigners,Level 1,,The aim of the course is to help learners unde...,006414c5-3520-43af-8125-967f9c787902.json



Examples — shorter but present (20–79 chars) — CLEANED subset:


,_name,_level,_country_final,_desc,_source_file
599,Diploma in Food and Beverage Service Operations,Level 4,,Typical programme duration is 2 Semesters + II...,03643875-ba22-4890-9fcc-8a72aed3ab14.json
2543,Award in Information Technology,Level 2,,The aim of this module is to provide basic IT ...,0ed990b8-ec20-47ec-be3d-89b34a19f757.json
6646,Diploma in Food preparation & Productions Oper...,Level 4,,"2 Semesters IITP (1 Year), Typical programme d...",27244619-6633-4d4f-85ad-9e67aee9d9ad.json
13881,Undergraduate Certificate in Sportsbook,Level 5,,Target Group: Those aiming for a career in Spo...,4fd12067-5bf2-4c56-a7c4-69e064626c8b.json
14079,Award in Human Resources Management (Online),Level 5,,Contact training provider for further information,50fd865b-6bf8-45ee-b598-cea82f605f9c.json
15380,Award in Budgeting,Level 2,,This unit will develop the skills of learners ...,58e5ba87-15e0-40a0-aa15-591e16a56d61.json
15741,Fertility and procreation specialist (Revised ...,Level 4,,The Fetility and procreation specialist works ...,5b05cb05-6af9-4c6f-b02b-03621090a3cb.json
16749,Higher National Diploma in Food and Beverage M...,Level 5,,"Typical programme duration is 2 Semesters, 2 S...",60b9f325-6ad7-4a6f-8b1f-bb3cdb72fa5a.json
17808,Award in Accounting,Level 3,,The programme intends to prepare individuals f...,66ecd762-2ad1-4e80-84a4-e35f3d386237.json
17984,Award in Real Estate Branch Manager,Level 4,,Dhalia Real Estate Agency Ltd’s new Real Estat...,67d50d66-982e-47e5-a655-d1d96c01d730.json



Saved: qualifications_essentials_with_desc_clean.parquet & .csv


#### 4.b) Inspect processed csv files

In [18]:
# --- Load cleaned CSV & run inspections ---

import os
import pandas as pd
import numpy as np

# Your base path (same folder where qualifications.csv lives)
BASE_PATH = "/Users/go82gax/Documents/Projekte/LFS/Analyse/Daten/QUAL Daten"
os.chdir(BASE_PATH)

# 1) Load CSV
csv_path = "qualifications.csv"
df = pd.read_csv(csv_path, low_memory=False)  # add dtype options if needed

# 2) Basic shape & schema
print(f"File: {csv_path}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]:,}\n")

print("Column list:")
for c in df.columns:
    print(f"  - {c}")

print("\nDtypes:")
display(df.dtypes.to_frame("dtype"))

# 3) Missingness (considering NaN/None)
miss_cnt = df.isna().sum()
miss_pct = (miss_cnt / len(df) * 100).round(2)
miss_df  = pd.DataFrame({"missing_count": miss_cnt, "missing_percent": miss_pct}) \
             .sort_values("missing_percent", ascending=False)
print("\nMissingness (per column):")
display(miss_df)

# 4) Cardinality (unique values per column)
card = df.nunique(dropna=True).sort_values(ascending=False)
print("\nCardinality (unique values per column):")
display(card.to_frame("unique_values"))

# 5) Low-cardinality categorical distributions (top 15 values)
obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
low_card = [c for c in obj_cols if 0 < df[c].nunique(dropna=True) <= 30]
if low_card:
    print("\nDistributions for low-cardinality columns (top 15 each):")
    vc_tables = []
    for c in low_card:
        vc = df[c].value_counts(dropna=False).head(15).to_frame(c)
        vc_tables.append(vc)
    display(pd.concat(vc_tables, axis=1))
else:
    print("\nNo low-cardinality object columns (<= 30 unique values).")

# 6) Numeric descriptives (if any)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if num_cols:
    print("\nNumeric descriptives:")
    display(df[num_cols].describe(percentiles=[.01,.05,.25,.5,.75,.9,.95,.99]).T)
else:
    print("\nNo numeric columns found.")

# 7) Text/description checks (optional, only if present)
desc_candidates = [c for c in ["_desc", "description", "learningOutcomeSummary.noteLiteral", "additionalNote"] if c in df.columns]
if desc_candidates:
    # Prefer _desc if you created it earlier; else fall back
    dcol = "_desc" if "_desc" in desc_candidates else desc_candidates[0]
    print(f"\nDescription column used for inspection: {dcol}")

    # Treat non-empty after stripping whitespace
    desc = df[dcol].astype(str)
    nonempty = desc.str.strip().ne("") & desc.str.lower().ne("nan")
    cov_any = int(nonempty.sum())
    cov20   = int((desc.str.len() >= 20).sum())
    total   = len(df)
    print(f"Description coverage: any: {cov_any:,} ({cov_any/total*100:.2f}%), "
          f">=20 chars: {cov20:,} ({cov20/total*100:.2f}%)")

    # Length stats
    lens = desc.str.len()
    print("\nDescription length stats (characters):")
    display(lens.describe(percentiles=[.01,.05,.25,.5,.75,.9,.95,.99]).to_frame("char_count").T)

    # Sample non-empty examples
    print("\nSample non-empty descriptions (first 10):")
    sample_cols = [c for c in ["_name","title","_level","EQFLevel.prefLabel","_country","_country_final","_source_file"] if c in df.columns]
    sample_cols = [dcol] + sample_cols  # put desc first
    display(df.loc[nonempty, sample_cols].head(10))
else:
    print("\nNo description-like columns found (looked for _desc, description, learningOutcomeSummary.noteLiteral, additionalNote).")

# 8) Quick peek at the first few rows
print("\nHead(3):")
display(df.head(3))


File: qualifications.csv
Rows: 10,223 | Columns: 6

Column list:
  - title
  - country
  - qualificationLevel
  - qualificationLevelNum
  - description
  - learningOutcome

Dtypes:


,dtype
title,object
country,object
qualificationLevel,object
qualificationLevelNum,int64
description,object
learningOutcome,object



Missingness (per column):


,missing_count,missing_percent
title,0,0.0
country,0,0.0
qualificationLevel,0,0.0
qualificationLevelNum,0,0.0
description,0,0.0
learningOutcome,0,0.0



Cardinality (unique values per column):


...

,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,max
char_count,10223.0,1010.892008,988.362578,1.0,101.0,212.0,467.0,806.0,1160.0,1970.0,2655.5,4808.24,19770.0



Sample non-empty descriptions (first 10):


,description,title
0,Students will be able to: demonstrate autonomo...,Bachelor's degree in theological studies and ...
1,The candidate is able to: plan and organize th...,Roma coordinator
2,This qualification is a dual vocational educat...,Investment fund specialist (m/f)
3,"Candidates will be able to: plan, prepare and ...",Basket maker
4,The Master in Teaching and Learning (MTL) is a...,Master in Teaching and Learning
5,This module explores the theories on Politics ...,"Award in Politics, Power and the State"
6,The candidate is able to: Electronics and Auto...,ELECTRONICS AND AUTOMATION TECHNICIAN
7,This course provides an up-to-date overview of...,Master of Business Administration (MBA) in Blo...
8,The aim of the course is to help learners unde...,Award in English Language for Foreigners
9,This module addresses strategies for effective...,Award in Testing Strategies (Academic Skills)



Head(3):


,title,country,qualificationLevel,qualificationLevelNum,description,learningOutcome
0,Bachelor's degree in theological studies and ...,Slovenia,Level 6,6,Students will be able to: demonstrate autonomo...,Students will be able to: demonstrate autonomo...
1,Roma coordinator,Slovenia,Level 4,4,The candidate is able to: plan and organize th...,The candidate is able to: plan and organize th...
2,Investment fund specialist (m/f),Germany,Level 4,4,This qualification is a dual vocational educat...,Learning outcomes — Manage securities accounts...


In [19]:
# --- Completeness checks for required fields ---

import pandas as pd
import numpy as np

# If you've already loaded df = pd.read_csv("qualifications.csv"), reuse it.
# Otherwise uncomment:
# df = pd.read_csv("qualifications.csv", low_memory=False)

REQUIRED = ["title", "country", "qualificationLevel", "description"]

def nonempty(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    return (s != "") & (s.str.lower() != "nan")

N = len(df)

# 1) Baseline completeness: all required columns non-empty
mask_complete = pd.Series(True, index=df.index)
for col in REQUIRED:
    if col not in df.columns:
        raise ValueError(f"Missing column in CSV: {col}")
    mask_complete &= nonempty(df[col])

complete_rows = int(mask_complete.sum())
pct_complete  = round(100 * complete_rows / N, 2)

print(f"Complete rows (title, country, qualificationLevel, description present): "
      f"{complete_rows:,} / {N:,} ({pct_complete}%)")

# 2) Stricter completeness: require description length >= 20
desc_len = df["description"].astype(str).str.len()
mask_complete20 = mask_complete & (desc_len >= 20)
complete20_rows = int(mask_complete20.sum())
pct_complete20  = round(100 * complete20_rows / N, 2)

print(f"Complete rows + description ≥ 20 chars: {complete20_rows:,} / {N:,} ({pct_complete20}%)")

# 3) Per-country breakdown for both definitions
def summarize(mask, label):
    out = (
        df.loc[mask, "country"]
          .value_counts()
          .rename("rows")
          .to_frame()
          .assign(percent=lambda x: (x["rows"] / mask.sum() * 100).round(2) if mask.sum() else 0)
    )
    out.index.name = f"country ({label})"
    return out

print("\nPer-country breakdown (baseline completeness):")
display(summarize(mask_complete, "complete"))

print("\nPer-country breakdown (complete + desc ≥ 20):")
display(summarize(mask_complete20, "complete+20"))

# 4) Optional: How many unique (title, country, qualificationLevel) among complete rows?
uniq_triplets = (
    df.loc[mask_complete, ["title", "country", "qualificationLevel"]]
      .drop_duplicates()
      .shape[0]
)
print(f"\nUnique (title, country, qualificationLevel) among complete rows: {uniq_triplets:,}")

# 5) Optional: quick look at a few complete rows
print("\nSample complete rows (first 10):")
display(df.loc[mask_complete, ["title","country","qualificationLevel","qualificationLevelNum","description"]].head(10))


Complete rows (title, country, qualificationLevel, description present): 10,223 / 10,223 (100.0%)
Complete rows + description ≥ 20 chars: 10,213 / 10,223 (99.9%)

Per-country breakdown (baseline completeness):


,rows,percent
country (complete),,
Malta,6701,65.55
Slovenia,2076,20.31
Germany,1325,12.96
Serbia,121,1.18



Per-country breakdown (complete + desc ≥ 20):


,rows,percent
country (complete+20),,
Malta,6694,65.54
Slovenia,2073,20.30
Germany,1325,12.97
Serbia,121,1.18



Unique (title, country, qualificationLevel) among complete rows: 8,286

Sample complete rows (first 10):


,title,country,qualificationLevel,qualificationLevelNum,description
0,Bachelor's degree in theological studies and ...,Slovenia,Level 6,6,Students will be able to: demonstrate autonomo...
1,Roma coordinator,Slovenia,Level 4,4,The candidate is able to: plan and organize th...
2,Investment fund specialist (m/f),Germany,Level 4,4,This qualification is a dual vocational educat...
3,Basket maker,Slovenia,Level 4,4,"Candidates will be able to: plan, prepare and ..."
4,Master in Teaching and Learning,Malta,Level 7,7,The Master in Teaching and Learning (MTL) is a...
5,"Award in Politics, Power and the State",Malta,Level 6,6,This module explores the theories on Politics ...
6,ELECTRONICS AND AUTOMATION TECHNICIAN,Serbia,Level 4,4,The candidate is able to: Electronics and Auto...
7,Master of Business Administration (MBA) in Blo...,Malta,Level 7,7,This course provides an up-to-date overview of...
8,Award in English Language for Foreigners,Malta,Level 1,1,The aim of the course is to help learners unde...
9,Award in Testing Strategies (Academic Skills),Malta,Level 5,5,This module addresses strategies for effective...
